In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Лабораторная работа: «Наведение порядка в каталоге загрузок».

Шаги:
  1. make_downloads()  — создаёт «захламлённый» каталог lab53/downloads;
  2. organize()        — строит план (dry-run) и выполняет разбор по типам/месяцам;
  3. remove_empty_dirs + журнал lab53/organize.log;
  4. report()          — отчёт по разобранному каталогу;
  5. проверка «ни один файл не потерян»;
  6. undo()            — откат перемещений по журналу и проверка восстановления.

Запуск:  python lab53.py
Двойной запуск подряд даёт одинаковый отчёт (генератор детерминирован).
"""

import os
import random
import shutil
from datetime import date, datetime, time, timedelta

# ------------------------------------------------------------------ константы
SEED = 2026
TODAY = date(2026, 11, 12)          # «сегодня» для всех расчётов
ROOT = "lab53"                      # рабочий каталог (создаётся заново)

TYPES = {
    "документы":   [".docx", ".pdf", ".txt"],
    "таблицы":     [".xlsx", ".csv"],
    "изображения": [".jpg", ".png"],
    "архивы":      [".zip"],
    "код":         [".py", ".ipynb"],
}
OTHER_TYPE = "прочее"               # всё, что не подошло (в т.ч. без расширения)

BASE_NAMES = ["report", "отчёт", "scan", "photo", "data", "lecture", "setup", "notes"]
ALL_EXTS = [ext for exts in TYPES.values() for ext in exts]
WEEKDAYS_RU = ["пн", "вт", "ср", "чт", "пт", "сб", "вс"]


# ------------------------------------------------------------------ утилиты
def fmt_mb(nbytes):
    """Объём в мебибайтах с двумя знаками после запятой."""
    return f"{nbytes / (1024 * 1024):.2f} МБ"


def file_type(name):
    """Тип файла по расширению; неизвестные и без расширения — «прочее»."""
    ext = os.path.splitext(name)[1].lower()
    for type_name, exts in TYPES.items():
        if ext in exts:
            return type_name
    return OTHER_TYPE


def snapshot(path):
    """
    Слепок дерева: список (относительный путь, размер, mtime с точностью до мс).
    Используется для проверок «до/после» и «каталог восстановлен».
    """
    result = []
    for dirpath, dirnames, filenames in os.walk(path):
        dirnames.sort()
        for name in sorted(filenames):
            full = os.path.join(dirpath, name)
            st = os.stat(full)
            result.append((os.path.relpath(full, path), st.st_size, round(st.st_mtime, 3)))
    return sorted(result)


# ------------------------------------------------------- шаг 1: захламление
def _random_name(rnd):
    """Случайное имя: основа [+ _1.._20] + расширение (10 % — .tmp или без него)."""
    name = rnd.choice(BASE_NAMES)
    if rnd.random() < 0.5:
        name += f"_{rnd.randint(1, 20)}"
    if rnd.random() < 0.10:
        name += rnd.choice([".tmp", ""])
    else:
        name += rnd.choice(ALL_EXTS)
    return name


def make_downloads(root=ROOT, n=50, seed=SEED):
    """
    Создаёт root/downloads с подкаталогами old и «Новая папка» и n файлов
    (первые пять — сразу в двух каталогах, т.е. всего файлов n + 5).
    Возвращает список созданных путей.
    """
    rnd = random.Random(seed)

    if os.path.isdir(root):                 # каждый запуск — с чистого листа
        shutil.rmtree(root)

    downloads = os.path.join(root, "downloads")
    subdirs = ["", "old", "Новая папка"]
    for sub in subdirs:
        os.makedirs(os.path.join(downloads, sub), exist_ok=True)

    created = []
    for i in range(n):
        # куда кладём файл
        places = [rnd.choice(subdirs)]
        if i < 5:                            # первые пять — дубли в двух каталогах
            places.append(rnd.choice([s for s in subdirs if s != places[0]]))

        # имя, свободное во всех выбранных каталогах
        while True:
            name = _random_name(rnd)
            if all(not os.path.exists(os.path.join(downloads, p, name)) for p in places):
                break

        size = rnd.randint(1024, 3 * 1024 * 1024)                 # 1 КБ … 3 МБ
        days_ago = rnd.randint(0, 180)                            # в пределах 180 дней
        moment = datetime.combine(
            TODAY - timedelta(days=days_ago),
            time(rnd.randrange(9, 19), rnd.randrange(60)),        # час дня — без краевых эффектов
        )
        ts = moment.timestamp()
        payload = os.urandom(size)                                # случайное содержимое

        for sub in places:
            path = os.path.join(downloads, sub, name)
            with open(path, "wb") as f:
                f.write(payload)
            os.utime(path, (ts, ts))
            created.append(path)

    return created


# ---------------------------------------------------------- шаг 2: разбор
def _free_name(path, used):
    """Свободное имя: report.docx → report (2).docx → report (3).docx …"""
    if path not in used and not os.path.exists(path):
        used.add(path)
        return path
    base, ext = os.path.splitext(path)
    i = 2
    while True:
        candidate = f"{base} ({i}){ext}"
        if candidate not in used and not os.path.exists(candidate):
            used.add(candidate)
            return candidate
        i += 1


def organize(src, dst, dry_run=True):
    """
    Строит/выполняет план разбора src → dst/<тип>/<ГГГГ-ММ>/<имя>.

    dry_run=True  — только печатает первые 5 строк плана и общее число файлов;
    dry_run=False — перемещает файлы (os.replace), ошибки OSError перехватывает.

    Возвращает список пар (откуда, куда).
    """
    used = set()                 # уже занятые имена в целевом дереве (в рамках плана)
    plan = []

    for dirpath, dirnames, filenames in os.walk(src):
        dirnames.sort()
        for name in sorted(filenames):
            full = os.path.join(dirpath, name)
            moment = datetime.fromtimestamp(os.path.getmtime(full))
            month = f"{moment.year:04d}-{moment.month:02d}"
            target_dir = os.path.join(dst, file_type(name), month)
            target = _free_name(os.path.join(target_dir, name), used)
            plan.append((full, target))

    if dry_run:
        for s, d in plan[:5]:
            print(f"[план] {s} -> {d}")
        print(f"... всего в плане: {len(plan)} файлов")
        return plan

    moved = []
    for s, d in plan:
        try:
            os.makedirs(os.path.dirname(d), exist_ok=True)
            os.replace(s, d)
            moved.append((s, d))
        except OSError as exc:
            print(f"[ошибка] {s}: {exc}")
    return moved


# ---------------------------------------------------------- шаг 3: уборка
def remove_empty_dirs(src):
    """Удаляет пустые каталоги внутри src (сам src не трогает)."""
    removed = 0
    for dirpath, dirnames, filenames in os.walk(src, topdown=False):
        if os.path.abspath(dirpath) == os.path.abspath(src):
            continue
        if not os.listdir(dirpath):
            os.rmdir(dirpath)
            removed += 1
    return removed


# ---------------------------------------------------------- шаг 4: отчёт
def report(dst):
    """Печатает отчёт по разобранному каталогу."""
    files = []
    for dirpath, dirnames, filenames in os.walk(dst):
        for name in filenames:
            full = os.path.join(dirpath, name)
            st = os.stat(full)
            files.append({
                "name": name,
                "path": full,
                "type": os.path.relpath(full, dst).split(os.sep)[0],
                "size": st.st_size,
                "mtime": datetime.fromtimestamp(st.st_mtime),
            })

    # по типам, упорядочено по объёму
    stats = {}
    for f in files:
        cnt, total = stats.get(f["type"], (0, 0))
        stats[f["type"]] = (cnt + 1, total + f["size"])

    for type_name, (cnt, total) in sorted(stats.items(), key=lambda kv: -kv[1][1]):
        print(f"{type_name:<12} {cnt:>2} файлов  {fmt_mb(total)}")

    # по месяцам
    months = {}
    for f in files:
        key = f"{f['mtime']:%Y-%m}"
        months[key] = months.get(key, 0) + 1
    print("По месяцам: " + ", ".join(f"{m} — {c}" for m, c in sorted(months.items())))

    # самый старый / самый новый
    def describe(f):
        return f"{f['mtime']:%d.%m.%Y}, {WEEKDAYS_RU[f['mtime'].weekday()]}"

    oldest = min(files, key=lambda f: (f["mtime"], f["name"]))
    newest = max(files, key=lambda f: (f["mtime"], f["name"]))
    print(f"Самый старый: {oldest['name']} ({describe(oldest)}); "
          f"самый новый: {newest['name']} ({describe(newest)})")

    # крупнее 1 МБ
    big = sorted((f for f in files if f["size"] > 1024 * 1024), key=lambda f: -f["size"])
    print(f"Крупнее 1 МБ: {len(big)} файлов:")
    for f in big:
        print(f"  {f['name']:<24} {fmt_mb(f['size']):>10}  ({f['type']})")

    # не изменялись больше 90 дней
    stale = [f for f in files if (TODAY - f["mtime"].date()).days > 90]
    print(f"Не изменялись больше 90 дней: {len(stale)} файлов")


# ------------------------------------------------------------- откат (undo)
def undo(log_name):
    """
    Возвращает файлы на прежние места по журналу «откуда;куда».
    Каталоги, удалённые на шаге уборки, создаются заново.
    Возвращает число восстановленных файлов.
    """
    with open(log_name, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    restored = 0
    for line in reversed(lines):                 # в обратном порядке перемещений
        src, dst = line.split(";", 1)
        if not os.path.exists(dst):
            continue
        parent = os.path.dirname(src)
        if parent:
            os.makedirs(parent, exist_ok=True)
        os.replace(dst, src)
        restored += 1
    return restored


# ------------------------------------------------------------------- сценарий
def main():
    downloads = os.path.join(ROOT, "downloads")
    sorted_dir = os.path.join(ROOT, "sorted")
    log_path = os.path.join(ROOT, "organize.log")

    print(f"SEED = {SEED}, TODAY = {TODAY:%d.%m.%Y}, ROOT = {ROOT}")

    # --- шаг 1
    created = make_downloads()
    before = snapshot(downloads)
    size_before = sum(s for _, s, _ in before)
    print(f"Создано {len(created)} файлов в {downloads} (3 каталога), "
          f"объём {fmt_mb(size_before)}")

    # --- шаг 2 (план и выполнение)
    plan = organize(downloads, sorted_dir, dry_run=True)
    moved = organize(downloads, sorted_dir, dry_run=False)
    errors = len(plan) - len(moved)

    # --- шаг 3 (уборка + журнал)
    removed_dirs = remove_empty_dirs(downloads)
    print(f"Перемещено: {len(moved)}, ошибок: {errors}, "
          f"удалено пустых каталогов: {removed_dirs}")

    with open(log_path, "w", encoding="utf-8") as f:
        for src, dst in moved:
            f.write(f"{src};{dst}\n")
    print(f"Журнал: {log_path}")

    # --- шаг 4 (отчёт)
    print("-" * 70)
    report(sorted_dir)
    print("-" * 70)

    # --- проверка: ничего не потеряно
    after = snapshot(sorted_dir)
    size_after = sum(s for _, s, _ in after)
    ok = (len(after) == len(before)) and (size_after == size_before)
    print(f"Проверка: до {len(before)} файлов / {fmt_mb(size_before)}, "
          f"после {len(after)} файлов / {fmt_mb(size_after)} — "
          f"{'совпадает' if ok else 'НЕ СОВПАДАЕТ'}")

    # --- дополнительно: откат по журналу
    restored = undo(log_path)
    remove_empty_dirs(sorted_dir)
    if os.path.isdir(sorted_dir) and not os.listdir(sorted_dir):
        os.rmdir(sorted_dir)

    back = snapshot(downloads)
    same = (back == before)
    print(f"Откат: возвращено {restored} файлов; каталог downloads "
          f"{'полностью восстановлен' if same else 'ОТЛИЧАЕТСЯ от исходного'}")


if __name__ == "__main__":
    main()

SEED = 2026, TODAY = 12.11.2026, ROOT = lab53
Создано 55 файлов в lab53\downloads (3 каталога), объём 84.54 МБ
[план] lab53\downloads\data.png -> lab53\sorted\изображения\2026-07\data.png
[план] lab53\downloads\data_5.ipynb -> lab53\sorted\код\2026-11\data_5.ipynb
[план] lab53\downloads\lecture.csv -> lab53\sorted\таблицы\2026-08\lecture.csv
[план] lab53\downloads\lecture_13.docx -> lab53\sorted\документы\2026-05\lecture_13.docx
[план] lab53\downloads\notes.py -> lab53\sorted\код\2026-10\notes.py
... всего в плане: 55 файлов
Перемещено: 55, ошибок: 0, удалено пустых каталогов: 2
Журнал: lab53\organize.log
----------------------------------------------------------------------
документы    15 файлов  22.78 МБ
код          11 файлов  19.48 МБ
таблицы      11 файлов  16.06 МБ
изображения   7 файлов  12.95 МБ
архивы        6 файлов  10.86 МБ
прочее        5 файлов  2.41 МБ
По месяцам: 2026-05 — 9, 2026-06 — 3, 2026-07 — 17, 2026-08 — 10, 2026-09 — 4, 2026-10 — 6, 2026-11 — 6
Самый старый: s